## **6. Key-Value RDD 연산**

### **Spark 애플리케이션 시작**

In [2]:
# [+] PySpark 임포트 및 설정
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster('local').setAppName('key-value-rdd-operation')
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 04:14:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **groupBy( )**
입력 함수를 기준으로 그룹핑

In [2]:
# [+] 리스트로부터 rdd 생성 [1, 1, 2, 3, 5, 8]
rdd = sc.parallelize([1, 1, 2, 3, 5, 8])

In [5]:
# [+] groupBy()를 이용한 홀수/짝수 그룹핑 -> groups
groups = rdd.groupBy(lambda x : x%2)

In [6]:
# [+] 그룹핑 결과를 리스트로 반환 -> res
res = groups.collect()

In [7]:
# [+] 그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

1 [1, 1, 3, 5]
0 [2, 8]


### **groupByKey( )**
Key 값을 기준으로 그룹핑

In [9]:
# [+] 리스트로부터 key-value rdd 생성 [('a', 1), ('b', 1), ('a', 1)]
rdd = sc.parallelize([('a', 1), ('b', 1), ('a', 1)])
rdd.collect()

[('a', 1), ('b', 1), ('a', 1)]

In [10]:
rdd.keys().collect()

['a', 'b', 'a']

In [11]:
# [+] groupByKey()를 이용한 그룹핑 및 리스트 반환
groups = rdd.groupByKey()

In [12]:
# [+] 그룹핑 결과를 리스트로 반환
res = groups.collect()

In [16]:
# [+]그룹핑 결과(res) 출력
for k, v in res:
    print (k,':', list(v))

a : [1, 1]
b : [1]


In [17]:
# len(): Python 객체의 길이를 리턴
value = [1, 1]
len(value)

2

In [39]:
# mapValues(), len()을 이용한 값 빈도 계산, mapValues(len) 하면 그룹별 값 빈도 계산
counts = groups.mapValues(len)

In [23]:
counts.collect()

[('a', 2), ('b', 1)]

### **groupBy( ) 예제**
1. 리스트로부터 RDD 객체 생성
    + 리스트: ['C', 'C++', 'Python', 'Java', 'C#']  
2. 원소의 첫 번째 문자 값을 기준으로 그룹핑
    + 기대 결과:  
        J ['Java']  
        C ['C', 'C++', 'C#']  
        P ['Python']  
3. 그룹핑 결과를 리스트 객체로 출력

In [29]:
# [+] 리스트로부터 RDD 객체 생성
rdd = sc.parallelize(['Java','C', 'C++', 'Pythonn',  'C#'])

In [30]:
# [+] 원소별 첫 번째 값(x[0])을 기준으로 그룹핑 -> groups
groups = rdd.groupBy(lambda x : x[0])

In [31]:
# [+] 그룹핑 결과를 리스트로 반환 -> res
res = groups.collect()

In [32]:
# [+] 그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

J ['Java']
C ['C', 'C++', 'C#']
P ['Pythonn']


### **groupByKey( ) 예제**
1. list 객체로부터 3개의 파티션을 갖는 Key-Value RDD 생성하기
    + ```getNumPatitions()```: RDD 객체의 파티션 수 출력
2. Key 별로 그룹핑
3. 그룹핑 결과 리스트로 반환
4. 리스트 출력

In [40]:
# list 객체로부터 3개의 파티션을 갖는 Key-Value RDD 생성하기

rdd = sc.parallelize([  # 과목 별 점수 데이터
    ('Math', 7), ('Math', 2), ('English', 7),
    ('Science', 7), ('English', 4), ('English', 9),
    ('Math', 8), ('Math', 3), ('English', 4),
    ('Science', 6), ('Science', 9), ('Science', 5)
], 3) # 파티션 수: 3

In [41]:
# getNumPartitions(): RDD 객체의 파티션 수를 확인하는 액션 메서드
rdd.getNumPartitions()

3

In [52]:
# [+] 파티션별 RDD 값 확인
rdd.glom().collect()

[[('Math', 7), ('Math', 2), ('English', 7), ('Science', 7)],
 [('English', 4), ('English', 9), ('Math', 8), ('Math', 3)],
 [('English', 4), ('Science', 6), ('Science', 9), ('Science', 5)]]

In [53]:
# [+] Key 별로 그룹핑 -> groups
groups = rdd.groupByKey()

In [54]:
# [+] 그룹핑 결과 리스트로 반환 -> res
res = groups.collect()

In [55]:
# 그룹핑 결과(res) 출력
for k, v in res:
    print(k, list(v))

Math [7, 2, 8, 3]
Science [7, 6, 9, 5]
English [7, 4, 9, 4]


In [56]:
# key별 그룹핑을 2개의 파티션으로 분리하여 수행 -> groups
groups = rdd.groupByKey(2)

In [57]:
# [+] 파티션 수 출력
groups.getNumPartitions()

2

In [63]:
# [+] 파티션별 RDD 그룹 확인
res = groups.glom().collect()
res

[[('Science', <pyspark.resultiterable.ResultIterable at 0x7a4e7af76180>)],
 [('Math', <pyspark.resultiterable.ResultIterable at 0x7a4e7af75eb0>),
  ('English', <pyspark.resultiterable.ResultIterable at 0x7a4e7af75f70>)]]

In [73]:
# [+] 0번째 파티션에 대한 그룹핑 결과(res)를 출력
# for k ,v in res[0]:
#    print (k, list(v))
# 이것도 가능
print(res[0][0][0], list(res[0][0][1]))

Science [7, 6, 9, 5]


26/10/08 05:11:27 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-4144cb66-9c5a-4600-9f9f-7cc255a9a288. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-4144cb66-9c5a-4600-9f9f-7cc255a9a288
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize

In [71]:
# [+] 1번째 파티션에 대한 그룹핑 결과(res)를 출력
for k ,v in res[1]:
    print (k, list(v))

Math [7, 2, 8, 3]
English [7, 4, 9, 4]


### **reduce( ) vs reduceByKey( )**
+ ```reduce()```: 입력 함수를 기준으로 집계를 수행하는 액션 메서드
+ ```reduceByKey()```: Key를 기준으로 그룹핑 및 집계를 수행하는 변환 메서드

In [3]:
# [+] reduce() 를 이용한 총합 구하기
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x, y : x+y)

15

In [6]:
# 리스트로부터 K-V RDD 생성 -> rdd
rdd = sc.parallelize([('a', 1), ('b', 1), ('a', 1)])

In [7]:
# [+] reduceByKey()를 이용한 Key별 총합 계산 및 리스트로 반환
rdd.reduceByKey(lambda x, y : x + y).collect()

[('a', 2), ('b', 1)]

In [10]:
# 과목 별 점수 데이터
rdd = sc.parallelize([
    ('Math', 7), ('Math', 2), ('English', 7),
    ('Science', 7), ('English', 4), ('English', 9),
    ('Math', 8), ('Math', 3), ('English', 4),
    ('Science', 6), ('Science', 9), ('Science', 5)
], 3)

In [11]:
# [+] reduceByKey() 를 이용한 과목 별 점수 총합 구하기
rdd.reduceByKey(lambda x,y : x+y).collect()

[('Math', 20), ('Science', 27), ('English', 24)]

### **mapValues( ) vs map( )**
+ ```map()```: key, value 전부 함수에 적용
+ ```mapValues()```: value에만 함수를 적용

In [12]:
# 리스트로부터 K-V RDD 객체 생성
rdd = sc.parallelize([
    ('a', ['apple', 'banana', 'lemon']),
    ('b', ['grapes'])
])

In [13]:
# mapValues(), len()를 이용하여 키 별 갯수 세기
rdd.mapValues(len).collect()

[('a', 3), ('b', 1)]

In [14]:
# map(): key, value 에 함수를 적용
rdd.map(len).collect()  # 결과가 mapValues()와 다른 이유는? key도 x로 두기 때문에 첫번째 a,[..., ..., ...] 두개, 두번째 b,[...] 두개 해서 2,2로 뜸

[2, 2]

### **countByKey( )**
Key별 값 개수를 계산 및 딕셔너리로 반환하는 액션 메서드

In [15]:
# countByKey()를 이용하여 Key별 값 배수 계산 및 반환
rdd = sc.parallelize([
    ('a', 1), ('b', 1), ('a', 1)
])

res = rdd.countByKey()

In [16]:
# countByKey() 결과 출력
print(res)

defaultdict(<class 'int'>, {'a': 2, 'b': 1})


### **keys( )**
key 목록을 RDD로 생성하는 변환 메서드

In [17]:
# 과목 별 점수 데이터
rdd = sc.parallelize([
    ('Math', 7), ('Math', 2), ('English', 7),
    ('Science', 7), ('English', 4), ('English', 9),
    ('Math', 8), ('Math', 3), ('English', 4),
    ('Science', 6), ('Science', 9), ('Science', 5)
], 3)



In [18]:
# RDD 키 목록 생성 및 출력
rdd.keys().collect()

['Math',
 'Math',
 'English',
 'Science',
 'English',
 'English',
 'Math',
 'Math',
 'English',
 'Science',
 'Science',
 'Science']

In [20]:
# [+] 유니크한 키 값 출력
rdd.keys().distinct().collect()

['Math', 'Science', 'English']

In [28]:
# [+] 유니크한 키 개수 출력
rdd.keys().distinct().count()

3

### **Join 연산**
+ ```join()```: 내부조인
+ ```leftOuterJoin()```: 외부조인(첫 번째 RDD 중심)
+ ```rightOuterJoin()```: 외부조인(두 번째 RDD 중심)
+ ```fullOuterJoin()```: 외부조인(Union 연산과 동일)

In [37]:
# K-V RDD 객체 두 개 생성 
rdd1 = sc.parallelize([("a", 1), ("b", 4)])
rdd2 = sc.parallelize([("a", 2)])

In [38]:
# Join 연산
rdd1.join(rdd2).collect()

[('a', (1, 2))]

In [26]:
# Left Outer Join 연산
rdd1.leftOuterJoin(rdd2).collect()

[('b', (4, None)), ('a', (1, 2))]

In [27]:
# Right Outer Join 연산
rdd1.rightOuterJoin(rdd2).collect()

[('a', (1, 2))]

In [30]:
rdd1.fullOuterJoin(rdd2).collect()

[('b', (4, None)), ('a', (1, 2))]

TypeError: 'RDD' object is not subscriptable